In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("1. Loading engineered features...")
df = pd.read_csv('../data/processed/features_for_ml.csv')
df['date'] = pd.to_datetime(df['date'])

# Train-test split (Last 30 days for testing)
max_date = df['date'].max()
test_start_date = max_date - pd.Timedelta(days=30)

train_data = df[df['date'] < test_start_date]
test_data = df[df['date'] >= test_start_date]

# Define features and target
features = ['day_of_week', 'month', 'is_weekend', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7']
target = 'sales'

X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

print("2. Training Global LightGBM Model (Poisson Objective)...")
model = lgb.LGBMRegressor(
    objective='poisson',
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print("3. Generating Predictions...")
y_pred = model.predict(X_test)

# Evaluation Functions
def calculate_wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

def calculate_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

wmape_lgbm = calculate_wmape(y_test, y_pred)
rmse_lgbm = calculate_rmse(y_test, y_pred)

print("-" * 50)
print("LIGHTGBM MODEL PERFORMANCE (Last 30 Days)")
print(f"WMAPE: {wmape_lgbm:.4f} ({wmape_lgbm*100:.2f}%)")
print(f"RMSE:  {rmse_lgbm:.4f}")
print("-" * 50)
print(f"Improvement over Baseline:")
print(f"WMAPE reduced by: {123.41 - (wmape_lgbm*100):.2f} percentage points")

1. Loading engineered features...
2. Training Global LightGBM Model (Poisson Objective)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001643 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 205
[LightGBM] [Info] Number of data points in the train set: 14800, number of used features: 6
[LightGBM] [Info] Start training from score -0.026427
3. Generating Predictions...
--------------------------------------------------
LIGHTGBM MODEL PERFORMANCE (Last 30 Days)
WMAPE: 1.0750 (107.50%)
RMSE:  2.1064
--------------------------------------------------
Improvement over Baseline:
WMAPE reduced by: 15.91 percentage points


In [2]:
import shap
import matplotlib.pyplot as plt
import joblib
import os

print("4. Generating SHAP Explainability Plot...")
# SHAP TreeExplainer for LightGBM
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Plot summary
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=features, show=False)
plt.title("SHAP Feature Importance: What drives the sales predictions?", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("5. Saving the trained model for the FastAPI Backend...")
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/lightgbm_retail_model.pkl')
print("Model successfully saved to ml_pipeline/models/lightgbm_retail_model.pkl")

ImportError: DLL load failed while importing _criterion: An Application Control policy has blocked this file.